# Replay-based learning and ranker limitations

The training curve comes from the recovered V14 archive. It displays the historical approximate AUC, computed on an example-level validation split. Those checkpoints cannot establish replay-level generalization. The Phase 3 ranking implementation uses replay-grouped splits and handles ties.

In [ ]:
from pathlib import Path
import csv
import json
from collections import Counter, defaultdict
import matplotlib.pyplot as plt

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "results/phase3/EXPERIMENT_MANIFEST.json").is_file())
results = root / "results"

def load_csv(relative):
    with (results / relative).open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


In [ ]:
history = load_csv("phase3/v14_training_curve.csv")
summary = json.loads((results / "ranker_v14_summary.json").read_text(encoding="utf-8"))
epochs = [int(r["epoch"]) for r in history]
fig, (auc_ax, loss_ax) = plt.subplots(1, 2, figsize=(10, 3.6))
for name in ("train_auc", "val_auc"):
    auc_ax.plot(epochs, [float(r[name]) for r in history], label=name)
for name in ("train_loss", "val_loss"):
    loss_ax.plot(epochs, [float(r[name]) for r in history], label=name)
auc_ax.set(xlabel="Epoch", ylabel="Recorded approximate AUC")
loss_ax.set(xlabel="Epoch", ylabel="Recorded weighted loss")
for ax in (auc_ax, loss_ax):
    ax.legend()
plt.tight_layout()
plt.show()
print("Archived examples:", summary["n"])
print("Historical best validation AUC:", summary["best_val_auc"])
print("Archived export coefficient:", summary["export_coefficient"])

The export coefficient was zero. The learned score therefore had no effect in that export. The historical feature matrix and inference weights are outside the release. The grouped trainer’s fixture is solely a demonstration of the interface.